# Ivey 2008 Discussion — Q&A Dataset Generation

**Source:** `buffett-2008.pdf` (136 KB)  
**Content:** Buffett's discussion with Dr. George Athanassakos and Ivey MBA/HBA students, March 31, 2008  
**Context:** This discussion happened during the 2008 financial crisis — six months before Lehman collapsed. Buffett's responses here are uniquely valuable for Adaptability and Risk Management, two labels that the Florida speech barely touched.

**Primary labels expected:** Adaptability, Risk Management, Psychology, Personal Life  

Overlapping sub-chunks may produce similar pairs — this is handled by novelty scoring here and semantic dedup in the assembly notebook.

In [1]:
import sys, re, statistics, importlib
from pathlib import Path
from collections import Counter

sys.path.insert(0, "..")
import pipeline.core
importlib.reload(pipeline.core)
from pipeline.core import (
    Chunk, QAPair, LABELS, extract_text,
    classify_chunks, generate_all, score_all,
    filter_by_quality, coverage_audit,
    export_csv, export_detailed,
    save_checkpoint, load_checkpoint,
)

## Stage 1: Chunking

This document has a clean `Question:` / `Buffett:` structure with ~10 Q&A exchanges. Unlike the Florida speech, `Buffett:` appears on its own line rather than inline. The title block and closing "End of Q&A" line are stripped before splitting.

The happiness response (pages 8–10) is the longest single answer at ~10,000 chars and gets sub-chunked into 3 pieces with overlap. Everything else fits under the 4000-char limit.

In [2]:
def _sub_chunk(text, source_file, section, max_chars=4000, overlap=500):
    """Split oversized text at paragraph boundaries with overlap for context continuity."""
    if len(text) <= max_chars:
        return [Chunk(text=text, source_file=source_file,
                      source_section=section, chunk_strategy="qa_exchange")]
    paragraphs = [p.strip() for p in text.split('\n') if len(p.strip()) > 30]
    sub_chunks, current, current_len = [], [], 0
    for para in paragraphs:
        if current_len + len(para) > max_chars and current:
            sub_chunks.append(Chunk(
                text='\n'.join(current), source_file=source_file,
                source_section=section, chunk_strategy="qa_exchange_subchunk"))
            overlap_paras, overlap_len = [], 0
            for p in reversed(current):
                if overlap_len + len(p) > overlap: break
                overlap_paras.insert(0, p)
                overlap_len += len(p)
            current = overlap_paras + [para]
            current_len = overlap_len + len(para)
        else:
            current.append(para)
            current_len += len(para)
    if current:
        sub_chunks.append(Chunk(
            text='\n'.join(current), source_file=source_file,
            source_section=section,
            chunk_strategy="qa_exchange_subchunk" if sub_chunks else "qa_exchange"))
    return sub_chunks


def chunk_ivey_2008(pdf_path):
    """Chunk the Ivey 2008 discussion using its Question/Buffett structure."""
    raw = extract_text(pdf_path)

    # Strip title block — everything before first Question:
    first_q = raw.find("Question:")
    if first_q > 0:
        raw = raw[first_q:]

    # Strip closing line
    raw = re.sub(r"End of Q&A.*$", "", raw, flags=re.DOTALL).strip()

    # Split on Question: boundaries
    parts = re.split(r'\n(?=Question:)', raw)

    chunks = []
    for part in parts:
        part = part.strip()
        if len(part) < 50:
            continue

        # Extract question text for the section name
        q_match = re.match(r'Question:\s*(.+?)(?:\n\s*\n|\nBuffett:)', part, re.DOTALL)
        section = q_match.group(1).strip()[:80] if q_match else part[:80]

        chunks.extend(_sub_chunk(part, "ivey_2008.pdf", section))

    # Merge runts back into previous chunk
    merged = []
    for chunk in chunks:
        if len(chunk.text) < 1000 and merged:
            merged[-1].text += '\n' + chunk.text
        else:
            merged.append(chunk)

    return merged

In [3]:
chunks = chunk_ivey_2008(Path("../sources/buffett-2008.pdf"))

print(f"Total chunks: {len(chunks)}\n")
for i, c in enumerate(chunks):
    print(f"  [{i:2d}] {c.source_section[:55]:55s} | {len(c.text):5d} chars | ~{len(c.text.split()):4d} words")

Total chunks: 12

  [ 0] What are your thoughts about the Chinese Stock market?  |  3471 chars | ~ 666 words
  [ 1] How can you make money without investing a lot of money |  3566 chars | ~ 696 words
  [ 2] With the abundance of stocks and companies out there I  |  3097 chars | ~ 557 words
  [ 3] What do you make of the current malaise in markets and  |  3196 chars | ~ 593 words
  [ 4] In the past you have talked about population concerns.  |  2391 chars | ~ 395 words
  [ 5] Why don’t you do philanthropy yourself and why would yo |  2170 chars | ~ 407 words
  [ 6] What are your thoughts about politics and investing? So |  2145 chars | ~ 385 words
  [ 7] Can you comment on the role of securities regulators in |  3950 chars | ~ 727 words
  [ 8] What is happiness? Are you happy?                       |  4045 chars | ~ 802 words
  [ 9] What is happiness? Are you happy?                       |  4038 chars | ~ 794 words
  [10] What is happiness? Are you happy?                       |  2089 c

## Stage 2: Classification

The 2008 crisis context should push several chunks toward Adaptability and Risk Management — labels that Florida couldn't cover. The Chinese stock market question should land on Strategy Development or Timing. The happiness question should land on Psychology or Personal Life.

In [4]:
classified = await classify_chunks(chunks)
save_checkpoint(classified, "ivey_classified")

Classifying 12 chunks (skipping 0 pre-labeled)
  10/12
  12/12

Label distribution:
  Personal Life: 5
  Strategy Development: 3
  Psychology: 2
  Adaptability: 1
  Risk Management: 1
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\ivey_classified.pkl


In [5]:
print(f"{'Label':25s} {'Conf':5s} | Section")
print("-" * 90)
for c in classified:
    conf = f"{c.confidence:.2f}" if c.confidence else "N/A"
    label = c.label or "UNCLASSIFIED"
    print(f"  {label:25s} {conf:5s} | {c.source_section[:55]}")

failed = [c for c in classified if c.label is None]
if failed:
    print(f"\n!! {len(failed)} chunks UNCLASSIFIED — will be skipped in generation")

Label                     Conf  | Section
------------------------------------------------------------------------------------------
  Psychology                0.85  | What are your thoughts about the Chinese Stock market?
  Personal Life             0.85  | How can you make money without investing a lot of money
  Strategy Development      0.90  | With the abundance of stocks and companies out there I 
  Adaptability              0.85  | What do you make of the current malaise in markets and 
  Personal Life             0.85  | In the past you have talked about population concerns. 
  Personal Life             0.85  | Why don’t you do philanthropy yourself and why would yo
  Strategy Development      0.85  | What are your thoughts about politics and investing? So
  Risk Management           0.85  | Can you comment on the role of securities regulators in
  Personal Life             0.95  | What is happiness? Are you happy?
  Psychology                0.95  | What is happiness? Are you

In [6]:
dist = Counter(c.label for c in classified if c.label)
print("Label distribution across chunks:\n")
for label, count in dist.most_common():
    bar = "█" * count
    print(f"  {label:25s} {count:2d} {bar}")

Label distribution across chunks:

  Personal Life              5 █████
  Strategy Development       3 ███
  Psychology                 2 ██
  Adaptability               1 █
  Risk Management            1 █


## Stage 3: Q&A Generation

12 chunks × 3 pairs = ~36 raw candidate pairs expected. Smaller yield than Florida (27 chunks) but the content per chunk is denser — Buffett is speaking during an active crisis, so the responses carry more weight for Adaptability and Risk Management.

In [7]:
pairs = await generate_all(classified, n_pairs=3)
save_checkpoint(pairs, "ivey_pairs_raw")
print(f"\nTotal raw pairs: {len(pairs)}")

  5/12 chunks | 15 pairs
  10/12 chunks | 30 pairs
  12/12 chunks | 36 pairs
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\ivey_pairs_raw.pkl

Total raw pairs: 36


In [8]:
seen_labels = set()
for p in pairs:
    if p.label in seen_labels:
        continue
    seen_labels.add(p.label)
    print(f"── {p.label} ──")
    print(f"  Q: {p.question}")
    print(f"  A: {p.answer[:250]}...")
    print(f"  Sublabel: {p.sublabel}")
    print()

── Psychology ──
  Q: How does Warren Buffett describe the ideal temperament for an investor when dealing with market fluctuations?
  A: Buffett describes the ideal temperament as one that views the market's volatility as an opportunity to be served, not as instruction. He uses the allegory of Mr. Market, a manic-depressive partner who offers to buy or sell his share of a business at ...
  Sublabel: temperament

── Personal Life ──
  Q: How did Warren Buffett's early life and habits contribute to his initial success in investing?
  A: Buffett's investing journey began with disciplined saving and self-education from a very young age. He saved money from age 6 to 11 to buy his first stock and had read many books on investing by that point. This early habit of saving, combined with c...
  Sublabel: early_life

── Strategy Development ──
  Q: How does Warren Buffett use the Value Line publication in his investment process?
  A: Buffett uses Value Line as a source to review thousands of com

## Stage 4: Quality Scoring & Filtering

Same setup as Florida — DeepSeek scores each pair on groundedness, label fit, richness, and novelty. The stricter prompt calibration from the Florida run carries over since we reloaded `core.py` at the top.

In [9]:
chunk_map = {c.chunk_id: c for c in classified}
scored = await score_all(pairs, chunk_map)
filtered = filter_by_quality(scored, threshold=0.7)
save_checkpoint(filtered, "ivey_pairs_filtered")

  Scored 10/36
  Scored 20/36
  Scored 30/36
  Scored 36/36
Quality filter: 25/36 passed (threshold=0.7)
Saved: C:\Users\Admin\OneDrive\Desktop\buffet-qa-pipeline\intermediate\ivey_pairs_filtered.pkl


In [10]:
scores = [p.composite_score for p in filtered if p.composite_score]
print(f"Pairs after filtering: {len(filtered)} / {len(pairs)} raw")
print(f"Drop rate: {(1 - len(filtered)/len(pairs))*100:.1f}%\n")
print(f"Score range: {min(scores):.2f} — {max(scores):.2f}")
print(f"Mean:   {statistics.mean(scores):.2f}")
print(f"Median: {statistics.median(scores):.2f}")

Pairs after filtering: 25 / 36 raw
Drop rate: 30.6%

Score range: 0.75 — 0.93
Mean:   0.83
Median: 0.84


## Stage 5: Coverage & Export

This document should fill the Adaptability gap from Florida. If it also contributes to Risk Management and Psychology, we're in strong shape for the first two sources combined.

In [11]:
print("Ivey 2008 contribution:\n")
report = coverage_audit(filtered)

Ivey 2008 contribution:

  Personal Life              10 pairs (40.0%)
    Sublabels hit: early_life, mentors, habits, personal_values
  Strategy Development        9 pairs (36.0%)
    Sublabels hit: value_investing_framework, margin_of_safety, business_quality, circle_of_competence, capital_allocation
  Timing                      0 pairs ( 0.0%)
  Risk Management             2 pairs ( 8.0%)
    Sublabels hit: leverage_avoidance, permanent_loss
  Adaptability                0 pairs ( 0.0%)
  Psychology                  4 pairs (16.0%)
    Sublabels hit: temperament, contrarian_thinking, rationality

  Total: 25 pairs


In [12]:
export_csv(filtered, Path("../intermediate/ivey_qa.csv"))
export_detailed(filtered, Path("../intermediate/ivey_qa_detailed.csv"))

Exported 25 pairs to ..\intermediate\ivey_qa.csv
  Personal Life: 10
  Strategy Development: 9
  Psychology: 4
  Risk Management: 2
Detailed export: ..\intermediate\ivey_qa_detailed.csv


In [13]:
label_counts = Counter(p.label for p in filtered)
print(f"FINAL: {len(filtered)} pairs across {len(label_counts)} labels\n")
for label, count in label_counts.most_common():
    best = max((p for p in filtered if p.label == label), key=lambda p: p.composite_score)
    print(f"── {label} ({count} pairs, best score: {best.composite_score:.2f}) ──")
    print(f"  Q: {best.question}")
    print(f"  A: {best.answer[:300]}")
    print()

FINAL: 25 pairs across 4 labels

── Personal Life (10 pairs, best score: 0.93) ──
  Q: How does Warren Buffett define happiness, and what personal example does he use to illustrate it?
  A: Buffett defines happiness as doing what you like with people you love. He states he is 'happy day after day' because he lives this principle, preferring his work to leisure activities like playing shuffleboard or being in Vegas. He contrasts this with people who have great wealth and public recognit

── Strategy Development (9 pairs, best score: 0.88) ──
  Q: In the case of ISCAR, how did Buffett's assessment of business quality influence his investment decision, independent of political considerations?
  A: Buffett's investment in ISCAR was driven by his assessment of its exceptional business quality, not politics. He describes it as "one of the most, if not the most, profitable businesses in Israel" and "a truly remarkable company." He was so impressed by the company's management and operations, b

In [14]:
# Idempotent reload — run this cell after any kernel restart
# Then skip to whichever stage you need to resume from

# classified = load_checkpoint("ivey_classified")
# pairs = load_checkpoint("ivey_pairs_raw")
# filtered = load_checkpoint("ivey_pairs_filtered")